In [ ]:
import numpy as np
import random
from collections import defaultdict


In [ ]:
class IndependentQLearning:
 
    
    def __init__(self, num_agents, state_space_size, action_space_size, 
                 learning_rate=0.1, discount_factor=0.95, epsilon_start=1.0, 
                 epsilon_end=0.1, epsilon_decay=0.995):

        self.num_agents = num_agents
        self.state_space_size = state_space_size
        self.action_space_size = action_space_size
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.epsilon = epsilon_start
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        
        self.q_tables = [defaultdict(lambda: np.zeros(action_space_size)) 
                         for _ in range(num_agents)]
        
        self.episode_rewards = [[] for _ in range(num_agents)]
        self.total_updates = [0 for _ in range(num_agents)]
    
    def state_to_key(self, state):
        return tuple(state.astype(int))
    
    def select_action(self, agent_idx, state, training=True):
        state_key = self.state_to_key(state)
        
        if training and random.random() < self.epsilon:
            return random.randint(0, self.action_space_size - 1)
        else:
            q_values = self.q_tables[agent_idx][state_key]
            max_q = np.max(q_values)
            best_actions = np.where(q_values == max_q)[0]
            return int(np.random.choice(best_actions))
    
    def update(self, agent_idx, state, action, reward, next_state, done):
        state_key = self.state_to_key(state)
        next_state_key = self.state_to_key(next_state)
        
        current_q = self.q_tables[agent_idx][state_key][action]
        
        if done:
            target_q = reward
        else:
            next_max_q = np.max(self.q_tables[agent_idx][next_state_key])
            target_q = reward + self.discount_factor * next_max_q
        
        new_q = current_q + self.learning_rate * (target_q - current_q)
        self.q_tables[agent_idx][state_key][action] = new_q
        
        self.total_updates[agent_idx] += 1
    
    def decay_epsilon(self):
        if self.epsilon > self.epsilon_end:
            self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    
    def get_q_value(self, agent_idx, state, action):
        state_key = self.state_to_key(state)
        return self.q_tables[agent_idx][state_key][action]
    
    def get_policy(self, agent_idx, state):
        state_key = self.state_to_key(state)
        q_values = self.q_tables[agent_idx][state_key]
        return int(np.argmax(q_values))
    
    def save_q_tables(self, filepath_prefix):
        import pickle
        for i in range(self.num_agents):
            filename = f"{filepath_prefix}_agent_{i}.pkl"
            with open(filename, 'wb') as f:
                pickle.dump(dict(self.q_tables[i]), f)
    
    def load_q_tables(self, filepath_prefix):
        import pickle
        for i in range(self.num_agents):
            filename = f"{filepath_prefix}_agent_{i}.pkl"
            try:
                with open(filename, 'rb') as f:
                    self.q_tables[i] = defaultdict(lambda: np.zeros(self.action_space_size), 
                                                    pickle.load(f))
            except FileNotFoundError:
                print(f"Warning: Q-table file {filename} not found")
